In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cpu


**Reading Data**

In [6]:
full_data = pd.read_csv("/content/drive/MyDrive/ColabNotebooks/full")
full_data.head()

,Unnamed: 0,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,...,Quarter,holiday_name,lag_1,lag_2,lag_4,lag_52,rolling_mean_4,rolling_mean_12,rolling_std_4,rolling_mean_52
0,0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,...,1,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,...,1,superbowl,24924.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,...,1,none,46039.49,24924.50,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,...,1,none,41595.55,46039.49,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,...,1,none,19403.54,41595.55,24924.5,NaN,32990.77,NaN,12832.106391,NaN


In [7]:
df = full_data[['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday']].copy()
df['Date'] = pd.to_datetime(df['Date'])
df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
df = df.rename(columns={'Weekly_Sales': 'y'})
df = df.sort_values(['unique_id', 'Date']).reset_index(drop=True)

# TimeSeriesDataSet needs an integer time index per series, not raw dates
df['time_idx'] = df.groupby('unique_id')['Date'].rank(method='first').astype(int) - 1

**Train Val Split**

In [8]:
test_horizon = 35

# find the actual last date available across train+val combined
full_dates = pd.to_datetime(full_data['Date']).sort_values().unique()
last_available_date = full_dates[-1]

# validation should end at last_available_date, and start 35 weeks before that
val_end_date = last_available_date
val_start_date = val_end_date - pd.Timedelta(weeks=test_horizon - 1)

print("New val range:", val_start_date, "to", val_end_date)

New val range: 2012-03-02 00:00:00 to 2012-10-26 00:00:00


In [9]:
df['Date'] = pd.to_datetime(df['Date'])

train_df = df[df['Date'] < val_start_date].copy()
val_df = df[df['Date'] >= val_start_date].copy()

print(train_df['Date'].max(), val_df['Date'].min())  # confirm no overlap
print(val_df['Date'].nunique())  # should now be 35

2012-02-24 00:00:00 2012-03-02 00:00:00
35


In [10]:
train_df.head()

,Store,Dept,Date,y,IsHoliday,unique_id,time_idx
0,10,1,2010-02-05,40212.84,False,10_1,0
1,10,1,2010-02-12,67699.32,True,10_1,1
2,10,1,2010-02-19,49748.33,False,10_1,2
3,10,1,2010-02-26,33601.22,False,10_1,3
4,10,1,2010-03-05,36572.44,False,10_1,4


In [11]:
val_df.head()

,Store,Dept,Date,y,IsHoliday,unique_id,time_idx
108,10,1,2012-03-02,30525.88,False,10_1,108
109,10,1,2012-03-09,33728.46,False,10_1,109
110,10,1,2012-03-16,34745.10,False,10_1,110
111,10,1,2012-03-23,38656.88,False,10_1,111
112,10,1,2012-03-30,46922.97,False,10_1,112


In [12]:
train_df.to_csv('/content/drive/MyDrive/WalmartPrices/NBeats/train_parquet.csv', index=False)
val_df.to_csv('/content/drive/MyDrive/WalmartPrices/NBeats/val_parquet.csv', index=False)